# 01 - Data exploration
**Goal:** understand the *Natural Images* dataset before any modelling: how many images per class, what they look like, how big they are, and whether any file is broken.

![pipeline](../docs/00_project_pipeline.png)

| Step | What we do |
|---|---|
| 1 | Count images per class (is the dataset balanced?) |
| 2 | Look at random samples of every class |
| 3 | Check image sizes and colour modes |
| 4 | Check that every file can be opened |
| 5 | Compute the *average image* of each class |

In [ ]:
import sys; sys.path.append("..")          # lets the notebook import src/common.py
from src.common import *
import matplotlib.pyplot as plt, numpy as np, pandas as pd
print("Project root:", ROOT)
print("QUICK mode:", QUICK)
from PIL import Image

## Step 1 - Class distribution

In [ ]:
counts = {c: len(list_images(c)) for c in CLASSES}
df_counts = pd.DataFrame({"class": list(counts), "images": list(counts.values())})
df_counts["share_%"] = (100 * df_counts["images"] / df_counts["images"].sum()).round(1)
display(df_counts)
print("Total images:", df_counts["images"].sum())

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(df_counts["class"], df_counts["images"], color=plt.cm.tab10.colors[:len(CLASSES)])
ax.bar_label(bars)
ax.set_title("Number of images per class"); ax.set_ylabel("images")
plt.xticks(rotation=30); plt.tight_layout()
save_fig("01_class_distribution.png"); plt.show()

## Step 2 - Random samples of every class

In [ ]:
rng = np.random.RandomState(SEED)
n_show = 6
fig, axes = plt.subplots(len(CLASSES), n_show, figsize=(2 * n_show, 2 * len(CLASSES)))
for r, cls in enumerate(CLASSES):
    files = list_images(cls)
    for c, i in enumerate(rng.choice(len(files), n_show, replace=False)):
        ax = axes[r, c]; ax.imshow(Image.open(files[i]).convert("RGB")); ax.axis("off")
        if c == 0: ax.set_title(cls, loc="left", fontsize=11)
plt.tight_layout(); save_fig("01_random_samples.png"); plt.show()

## Step 3 - Image sizes and colour modes
The CNN/MLP/HOG all need a *fixed* input size, so we must know how much the sizes vary.

In [ ]:
rows = []
for cls in CLASSES:
    for f in list_images(cls)[:300]:            # first 300 per class is enough for statistics
        with Image.open(f) as im:               # opening is lazy: only the header is read
            rows.append((cls, im.width, im.height, im.mode))
df_sizes = pd.DataFrame(rows, columns=["class", "width", "height", "mode"])
display(df_sizes.groupby("class")[["width", "height"]].agg(["min", "mean", "max"]).round(0))
print(df_sizes["mode"].value_counts())

fig, ax = plt.subplots(figsize=(6, 5))
for i, cls in enumerate(CLASSES):
    d = df_sizes[df_sizes["class"] == cls]
    ax.scatter(d["width"], d["height"], s=10, alpha=.5, label=cls, color=plt.cm.tab10.colors[i])
ax.set_xlabel("width (px)"); ax.set_ylabel("height (px)"); ax.legend(fontsize=7)
ax.set_title("Image sizes"); plt.tight_layout(); save_fig("01_image_sizes.png"); plt.show()

## Step 4 - Integrity check

In [ ]:
corrupt = []
for cls in CLASSES:
    for f in list_images(cls):
        try:
            with Image.open(f) as im: im.verify()
        except Exception:
            corrupt.append(f)
print("Unreadable files:", len(corrupt))
for f in corrupt[:10]: print("  ", f)

## Step 5 - Average image of each class

In [ ]:
fig, axes = plt.subplots(1, len(CLASSES), figsize=(2 * len(CLASSES), 2.4))
for ax, cls in zip(axes, CLASSES):
    arr = [np.asarray(Image.open(f).convert("RGB").resize((64, 64)), dtype=np.float32)
           for f in list_images(cls)[:200]]
    ax.imshow(np.mean(arr, axis=0) / 255); ax.set_title(cls, fontsize=9); ax.axis("off")
plt.suptitle("Mean image per class (first 200 images)"); plt.tight_layout()
save_fig("01_mean_images.png"); plt.show()

## Conclusions
* The dataset has **8 classes**; check the table above for the balance (it is only mildly imbalanced).
* Images have **different sizes** -> they must be resized to one common size (64 x 64) in the next notebook.
* Mean images show that colour is already informative for some classes (e.g. blue sky for *airplane*), which is why the deep-learning models use RGB while HOG uses grayscale *shape* information.